In [5]:
# %%
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.preprocessing import (
    OneHotEncoder,
    StandardScaler,
    LabelEncoder,
)

from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import f1_score


# =========================
# 1. DATA
# =========================
train = pd.read_csv("train.csv")

TARGET = "Demand_Category"

train = train.dropna()

X = train.drop(columns=[TARGET, "Date"])
y_raw = train[TARGET]


# =========================
# 2. ENCODE TARGET
# =========================
le = LabelEncoder()

y = le.fit_transform(y_raw)


# =========================
# 3. COLUMN SPLIT
# =========================
num_cols = X.select_dtypes(include=["int64", "float64"]).columns.tolist()

cat_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()


# =========================
# 4. PREPROCESSOR
# =========================
numeric_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    [
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore")),
    ]
)

preprocessor = ColumnTransformer(
    [
        ("num", numeric_transformer, num_cols),
        ("cat", categorical_transformer, cat_cols),
    ]
)


# =========================
# 5. PIPELINE
# =========================
pipeline = Pipeline(
    [
        ("prep", preprocessor),
        ("model", LogisticRegression(max_iter=2000)),
    ]
)


# =========================
# 6. CROSS VALIDATION
# =========================
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(
    pipeline,
    X,
    y,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
)

print("\nCV Macro F1:", scores.mean())


# =========================
# 7. FINAL MODEL
# =========================
pipeline.fit(X, y)

final_pipeline = pipeline

print("\nModel trained successfully")


CV Macro F1: 0.9307320503042756

Model trained successfully


# Submission

In [6]:
# %%
import pandas as pd


# =========================
# 8. TEST PREDICTIONS
# =========================
test = pd.read_csv("test.csv")

test_X = test.drop(columns=["Date"])

preds = final_pipeline.predict(test_X)

# convert labels back
pred_labels = le.inverse_transform(preds)


# =========================
# 9. SUBMISSION
# =========================
submission = pd.read_csv("sample_submission.csv")

submission["Demand_Category"] = pred_labels

submission.to_csv("my_first_submission.csv", index=False)

print("submission created successfully")
print(submission.head())

submission created successfully
   Kaggle_ID  Demand_Category
0          0                0
1          1                0
2          2                0
3          3                0
4          4                0
